# Inserting ISS Data To MySQL with SQLAlchemy

## Collecting data from the API

In [19]:
import json
import sqlalchemy
import pandas as pd
import requests
from datetime import datetime, date, timedelta
from pytz import timezone

In [20]:
%pip install requests
%pip install pandas
%pip install sqlalchemy
%pip install pymysql

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [21]:
url = "https://aerodatabox.p.rapidapi.com/airports/search/location/52.31/13.24/km/50/16"

querystring = {"withFlightInfoOnly":"true"}

headers = {
	"X-RapidAPI-Key": "6ba9930870msh47ec68ae33bba5fp188080jsn9a1dce38bbc5",
	"X-RapidAPI-Host": "aerodatabox.p.rapidapi.com"
}

response = requests.request("GET", url, headers=headers, params=querystring)

print(response.text)

{"searchBy":{"lat":52.31,"lon":13.24},"items":[{"icao":"EDDB","iata":"BER","name":"Berlin, Berlin Brandenburg","shortName":"Brandenburg","municipalityName":"Berlin","location":{"lat":52.35139,"lon":13.493889},"countryCode":"DE"}]}


In [22]:
airoports_list=['BER','HAM','MAN','LHR','BCN']
list_for_air=[]

for i in airoports_list:
  url = f"https://aerodatabox.p.rapidapi.com/airports/iata/{i}"

  headers = {
	"X-RapidAPI-Key": "6ba9930870msh47ec68ae33bba5fp188080jsn9a1dce38bbc5",
	"X-RapidAPI-Host": "aerodatabox.p.rapidapi.com"
  }

  response = requests.request("GET", url, headers=headers)
  list_for_air.append(response.json())
print(list_for_air)

[{'icao': 'EDDB', 'iata': 'BER', 'shortName': 'Brandenburg', 'fullName': 'Berlin, Berlin Brandenburg', 'municipalityName': 'Berlin', 'location': {'lat': 52.35139, 'lon': 13.493889}, 'country': {'code': 'DE', 'name': 'Germany'}, 'continent': {'code': 'EU', 'name': 'Europe'}, 'timeZone': 'Europe/Berlin', 'urls': {'webSite': 'https://ber.berlin-airport.de/', 'wikipedia': 'https://en.wikipedia.org/wiki/Berlin_Brandenburg_Airport', 'twitter': 'http://twitter.com/berlinairport', 'googleMaps': 'https://www.google.com/maps/@52.351389,13.493889,14z', 'flightRadar': 'https://www.flightradar24.com/52.35,13.49/14'}}, {'icao': 'EDDH', 'iata': 'HAM', 'shortName': 'Hamburg', 'fullName': 'Hamburg', 'municipalityName': 'Hamburg', 'location': {'lat': 53.6304, 'lon': 9.988229}, 'country': {'code': 'DE', 'name': 'Germany'}, 'continent': {'code': 'EU', 'name': 'Europe'}, 'timeZone': 'Europe/Berlin', 'urls': {'webSite': 'https://www.hamburg-airport.de/en/', 'wikipedia': 'https://en.wikipedia.org/wiki/Hambur

In [23]:
cities = pd.json_normalize(list_for_air)
cities

,icao,iata,shortName,fullName,municipalityName,timeZone,location.lat,location.lon,country.code,country.name,continent.code,continent.name,urls.webSite,urls.wikipedia,urls.twitter,urls.googleMaps,urls.flightRadar
0,EDDB,BER,Brandenburg,"Berlin, Berlin Brandenburg",Berlin,Europe/Berlin,52.35139,13.493889,DE,Germany,EU,Europe,https://ber.berlin-airport.de/,https://en.wikipedia.org/wiki/Berlin_Brandenbu...,http://twitter.com/berlinairport,"https://www.google.com/maps/@52.351389,13.4938...","https://www.flightradar24.com/52.35,13.49/14"
1,EDDH,HAM,Hamburg,Hamburg,Hamburg,Europe/Berlin,53.63040,9.988229,DE,Germany,EU,Europe,https://www.hamburg-airport.de/en/,https://en.wikipedia.org/wiki/Hamburg_Airport,http://twitter.com/HamburgAirport,"https://www.google.com/maps/@53.630401,9.98822...","https://www.flightradar24.com/53.63,9.99/14"
2,EGCC,MAN,Manchester,Manchester,Manchester,Europe/London,53.35370,-2.274950,GB,United Kingdom,EU,Europe,http://www.manchesterairport.co.uk/,https://en.wikipedia.org/wiki/Manchester_Airport,http://twitter.com/manairport,"https://www.google.com/maps/@53.353698,-2.2749...","https://www.flightradar24.com/53.35,-2.27/14"
3,EGLL,LHR,Heathrow,"London, London Heathrow",London,Europe/London,51.47060,-0.461941,GB,United Kingdom,EU,Europe,http://www.heathrow.com/,https://en.wikipedia.org/wiki/London_Heathrow_...,http://twitter.com/HeathrowAirport,"https://www.google.com/maps/@51.470600,-0.4619...","https://www.flightradar24.com/51.47,-0.46/14"
4,LEBL,BCN,Barcelona,Barcelona,Barcelona,Europe/Madrid,41.29710,2.078459,ES,Spain,EU,Europe,http://www.aena.es/,https://en.wikipedia.org/wiki/Barcelona_Intern...,http://twitter.com/Aena,"https://www.google.com/maps/@41.297100,2.07845...","https://www.flightradar24.com/41.30,2.08/14"


In [24]:
def icao_airport_codes(latitudes, longitudes):

  #assert len(latitudes) == len(longitudes)
  
  list_for_df = []

  for i in range(len(latitudes)):

    url = f"https://aerodatabox.p.rapidapi.com/airports/search/location/{latitudes[i]}/{longitudes[i]}/km/50/16"

    querystring = {"withFlightInfoOnly":"true"}

    headers = {
      "X-RapidAPI-Host": "aerodatabox.p.rapidapi.com",
      "X-RapidAPI-Key": "6ba9930870msh47ec68ae33bba5fp188080jsn9a1dce38bbc5"
    }

    response = requests.request("GET", url, headers=headers, params=querystring)

    list_for_df.append(pd.json_normalize(response.json()['items']))

  return pd.concat(list_for_df, ignore_index=True)

In [25]:
cities['location.lat'].to_list()

[52.35139, 53.6304, 53.3537, 51.4706, 41.2971]

In [26]:
cities['location.lon'].to_list()

[13.493889, 9.988229, -2.27495, -0.461941, 2.078459]

In [27]:
airports=icao_airport_codes(cities["location.lat"].to_list(), cities["location.lon"].to_list())
airports = airports.drop (index= 3 )


In [28]:
airports = airports.drop (index= 6 )

In [29]:
all_icao=airports['icao'].to_list()

In [30]:
def tomorrows_flight_arrivals(icao_list):

  today = datetime.now().astimezone(timezone('Europe/Berlin')).date()
  tomorrow = (today + timedelta(days=1))

  list_for_df = []

  for icao in icao_list:
    times = [["00:00","11:59"],["12:00","23:59"]]

    for time in times:
      url = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{icao}/{tomorrow}T{time[0]}/{tomorrow}T{time[1]}"
      querystring = {"withLeg":"true","direction":"Arrival","withCancelled":"false","withCodeshared":"true","withCargo":"false","withPrivate":"false"}
      headers = {
          'x-rapidapi-host': "aerodatabox.p.rapidapi.com",
          'x-rapidapi-key': "6ba9930870msh47ec68ae33bba5fp188080jsn9a1dce38bbc5"
          }
      response = requests.request("GET", url, headers=headers, params=querystring)
      flights_json = response.json()

      for flight in flights_json['arrivals']:
        flights_dict = {}
        flights_dict['arrival_icao'] = icao
        # .get() is another way of ensuring our code doesn't break
        # in the previous 2 notebooks you learnt about 'if' (cities) and 'try/except' (weather)
        # .get() works similar, it will get the text if possible, if there is no text a None value will be inserted instead
        flights_dict['arrival_time_local'] = flight['arrival'].get('scheduledTimeLocal', None)
        #flights_dict['arrival_terminal'] = flight['arrival'].get('terminal', None)
        #flights_dict['departure_city'] = flight['departure']['airport'].get('name', None)
        flights_dict['departure_icao'] = flight['departure']['airport'].get('icao', None)
        #flights_dict['departure_time_local'] = flight['departure'].get('scheduledTimeLocal', None)
        #flights_dict['airline'] = flight['airline'].get('name', None)
        flights_dict['flight_number'] = flight.get('number', None)
        #flights_dict['data_retrieved_on'] = datetime.now().astimezone(timezone('Europe/Berlin')).date()
        list_for_df.append(flights_dict)

  return pd.DataFrame(list_for_df)

In [31]:
flights_df=tomorrows_flight_arrivals(all_icao)

flights_df.reset_index(inplace=True)

flights_df.rename(columns={"index": "flight_id"})

,flight_id,arrival_icao,arrival_time_local,departure_icao,flight_number
0,0,EDDB,2023-02-14 07:05+01:00,KJFK,DL 92
1,1,EDDB,2023-02-14 07:30+01:00,EDDK,EW 12
2,2,EDDB,2023-02-14 07:50+01:00,EDDS,EW 2002
3,3,EDDB,2023-02-14 07:35+01:00,EDDL,EW 9048
4,4,EDDB,2023-02-14 07:55+01:00,EDDF,LH 170
...,...,...,...,...,...
3651,3651,LEBL,2023-02-14 22:15+01:00,LSZH,VY 6249
3652,3652,LEBL,2023-02-14 22:45+01:00,LIMC,VY 6337
3653,3653,LEBL,2023-02-14 22:20+01:00,EGKK,VY 7825
3654,3654,LEBL,2023-02-14 22:20+01:00,LEMH,VY 3719


In [32]:
schema="gans"   # name of the database you want to use here
host="127.0.0.1"        # to connect to your local server
user="root"
password="asd7ab8BG766B6LOhygs" # your password!!!!
port=3306
con = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'

In [37]:
flights_df.to_sql("flights",
            if_exists="append",
            con=con,
            index=False)

3656

For amazon cloud

In [38]:
schema="gans"   
host="wbs-project3-db.cfgm5ahdptv9.us-east-1.rds.amazonaws.com"        
user="admin"
password="kazxed-jeKhiv-8futwo" 
port=3306
con = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'

In [39]:
flights_df.to_sql("flights",
            if_exists="append",
            con=con,
            index=False)

3656